In [ ]:
# llm_exercise 프로젝트에서 진행

# 1. Qwen2.5-1.5B-Instruct

In [ ]:
# https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct
# 라이브러리 설치 transformers accelerate

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
###############################################################################################################
prompt = "Give me a short introduction to large language model."
messages = [
    {
        "role": "system",
        "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.",
    },
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
###############################################################################################################
generated_ids = model.generate(**model_inputs, max_new_tokens=512)
generated_ids = [
    output_ids[len(input_ids) :]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


In [18]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="auto", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [19]:
prompt = "hi"
messages = [
    {
        "role": "system",
        "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.",
    },
    {"role": "user", "content": prompt},
]
print(messages)
print("=" * 100)
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(text)
print("=" * 100)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
print(model_inputs)

[{'role': 'system', 'content': 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.'}, {'role': 'user', 'content': 'hi'}]
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
hi<|im_end|>
<|im_start|>assistant

{'input_ids': tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   6023, 151645,    198,
         151644,  77091,    198]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]])}


In [20]:
generated_ids = model.generate(**model_inputs, max_new_tokens=512)
print(generated_ids)
generated_ids = [
    output_ids[len(input_ids) :]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
print(generated_ids)
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

tensor([[151644,   8948,    198,   2610,    525,   1207,  16948,     11,   3465,
            553,  54364,  14817,     13,   1446,    525,    264,  10950,  17847,
             13, 151645,    198, 151644,    872,    198,   6023, 151645,    198,
         151644,  77091,    198,   9707,      0,   2585,    646,    358,   7789,
            498,   3351,     30, 151645]])
[tensor([  9707,      0,   2585,    646,    358,   7789,    498,   3351,     30,
        151645])]
Hello! How can I assist you today?


In [21]:
# messages -> response 까지 나오는 과정 파악하기
# 1_0_huggingface.ipynb에서 다룬 kanana로 다시 확인해보기

# 2. Kanana-nano-2.1b-instruct

In [22]:
# https://huggingface.co/kakaocorp/kanana-nano-2.1b-instruct
# 모델 에러 해결 - 주로 trasnformers의 버전 문제일 가능성이 높다.
# uv remove transformers
# uv add transformers==4.45.0

In [23]:
import transformers

print(transformers.__version__)

4.50.0


In [24]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "kakaocorp/kanana-nano-2.1b-instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [25]:
# 질문 바로 입력 → 답변 생성
messages = [
    {"role": "system", "content": "You are a helpful AI assistant developed by Kakao."},
    {"role": "user", "content": "한국의 수도는 어디야? 간단히 알려줘."},
]

model_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

In [26]:
with torch.no_grad():
    output = model.generate(
        **model_inputs,
        max_new_tokens=100,
        do_sample=False,
    )

# 입력 부분 제외하고 생성된 답변만 출력
response = tokenizer.decode(
    output[0][model_inputs["input_ids"].shape[-1] :], skip_special_tokens=True
)
print("[질문] 한국의 수도는 어디야? 간단히 알려줘.")
print(f"[답변] {response}")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[질문] 한국의 수도는 어디야? 간단히 알려줘.
[답변] 한국의 수도는 서울입니다.


# Gemma-3-1b-it

In [27]:
# https://huggingface.co/google/gemma-3-1b-it
# 모델 불러오기 에러 해결 과정
# uv remove transformers
# uv add transformers==4.50.0
# uv add bitsandbytes

In [28]:
from huggingface_hub import login
import os

login(token=os.getenv("HUGGINGFACE_TOKEN"))

In [29]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="google/gemma-3-1b-it",
    device="cuda",
    torch_dtype=torch.bfloat16,
)

messages = [
    [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "You are a helpful assistant."},
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Write a poem on Hugging Face, the company"},
            ],
        },
    ],
]

output = pipe(messages, max_new_tokens=50)

Device set to use cuda


In [30]:
print(output)

[[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Write a poem on Hugging Face, the company'}]}, {'role': 'assistant', 'content': 'Okay, here’s a poem about Hugging Face, aiming to capture its spirit and impact:\n\n**The Neural Bloom**\n\nIn realms of code, a curious quest,\nHugging Face, a haven, truly blessed.\nA community'}]}]]


In [31]:
for out in output:
    print(out)

[{'generated_text': [{'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]}, {'role': 'user', 'content': [{'type': 'text', 'text': 'Write a poem on Hugging Face, the company'}]}, {'role': 'assistant', 'content': 'Okay, here’s a poem about Hugging Face, aiming to capture its spirit and impact:\n\n**The Neural Bloom**\n\nIn realms of code, a curious quest,\nHugging Face, a haven, truly blessed.\nA community'}]}]


In [32]:
from transformers import AutoTokenizer, BitsAndBytesConfig, Gemma3ForCausalLM
import torch

model_id = "google/gemma-3-1b-it"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = Gemma3ForCausalLM.from_pretrained(
    model_id, quantization_config=quantization_config
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)

`low_cpu_mem_usage` was None, now default to True since model is quantized.


In [33]:
messages = [
    [
        {
            "role": "system",
            "content": [
                {"type": "text", "text": "You are a helpful assistant."},
            ],
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Write a poem on Hugging Face, the company"},
            ],
        },
    ],
]
inputs = (
    tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    .to(model.device)
    .to(torch.bfloat16)
)

Attempting to cast a BatchEncoding to type torch.bfloat16. This is not supported.


In [34]:
with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=64)

outputs = tokenizer.batch_decode(outputs)

In [35]:
print(outputs)

['<bos><start_of_turn>user\nYou are a helpful assistant.\n\nWrite a poem on Hugging Face, the company<end_of_turn>\n<start_of_turn>model\nOkay, here’s a poem about Hugging Face, aiming to capture its essence and spirit:\n\n**The Neural Forge**\n\nA world of models, vast and deep,\nWhere algorithms softly sleep,\nThen wake to life, a vibrant hue,\nWith Hugging Face, a helping view.\n\nFrom']


In [36]:
for out in outputs:
    print(out)

<bos><start_of_turn>user
You are a helpful assistant.

Write a poem on Hugging Face, the company<end_of_turn>
<start_of_turn>model
Okay, here’s a poem about Hugging Face, aiming to capture its essence and spirit:

**The Neural Forge**

A world of models, vast and deep,
Where algorithms softly sleep,
Then wake to life, a vibrant hue,
With Hugging Face, a helping view.

From


# 과제

In [37]:
# 1. LLM Instruct의 모델을 Input에서 model을 거쳐 Output까지 나오는 과정을 설명해주세요
# 2. 각각의 모델마다 오류를 겪었습니다. 미리 각 모델마다 주의해야 하는 것. 추가적으로 설치해야 하는 라이브러리를 모델카드로 만들어주세요.

# 파인튜닝 데이터셋은 어떻게 생겼을까?

In [38]:
# messages = [
#     [
#         {
#             "role": "system",
#             "content": [{"type": "text", "text": "You are a helpful assistant."},]
#         },
#         {
#             "role": "user",
#             "content": [{"type": "text", "text": "Write a poem on Hugging Face, the company"},]
#         },
#     ],
# ]

## 1) Dataset 이해하기

In [39]:
# 라이브러리 설치 uv add datasets
from datasets import load_dataset

dataset = load_dataset("beomi/KoAlpaca-v1.1a")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'url'],
        num_rows: 21155
    })
})

In [40]:
dataset["train"]

Dataset({
    features: ['instruction', 'output', 'url'],
    num_rows: 21155
})

In [41]:
dataset["train"][0]

{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?',
 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268'}

## 2) datasets 기능으로 데이터셋 만들기

In [42]:
sampledata = dataset["train"].select(range(10))
sampledata

Dataset({
    features: ['instruction', 'output', 'url'],
    num_rows: 10
})

In [43]:
sampledata[0]

{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?',
 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268'}

In [44]:
# data가 들어왔을 때 messages = [{"role":, "content":},{"role":, "content":}] 형태로 만드는 함수 만들기
def make_prompt(data):
    return {
        "result": [
            {"role": "user", "content": data["instruction"]},
            {"role": "user", "content": data["output"]},
        ]
    }

In [45]:
make_prompt(sampledata[0])

{'result': [{'role': 'user', 'content': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?'},
  {'role': 'user',
   'content': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.'}]}

In [46]:
test = sampledata.map(make_prompt)
test

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'output', 'url', 'result'],
    num_rows: 10
})

In [47]:
test[0]

{'instruction': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?',
 'output': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
 'url': 'https://kin.naver.com/qna/detail.naver?d1id=11&dirId=1116&docId=55320268',
 'result': [{'content': '양파는 어떤 식물 부위인가요? 그리고 고구마는 뿌리인가요?', 'role': 'user'},
  {'content': '양파는 잎이 아닌 식물의 줄기 부분입니다. 고구마는 식물의 뿌리 부분입니다. \n\n식물의 부위의 구분에 대해 궁금해하는 분이라면 분명 이 질문에 대한 답을 찾고 있을 것입니다. 양파는 잎이 아닌 줄기 부분입니다. 고구마는 다른 질문과 답변에서 언급된 것과 같이 뿌리 부분입니다. 따라서, 양파는 식물의 줄기 부분이 되고, 고구마는 식물의 뿌리 부분입니다.\n\n 덧붙이는 답변: 고구마 줄기도 볶아먹을 수 있나요? \n\n고구마 줄기도 식용으로 볶아먹을 수 있습니다. 하지만 줄기 뿐만 아니라, 잎, 씨, 뿌리까지 모든 부위가 식용으로 활용되기도 합니다. 다만, 한국에서는 일반적으로 뿌리 부분인 고구마를 주로 먹습니다.',
   'role': 'user'}]}